# Quill — Gemma 3 270M LoRA on CoEdIT (Colab T4)

Self-contained. Open in VS Code with the Google Colab extension, connect to a T4 runtime, then **Run All**.

**What this notebook does**
1. Verifies a CUDA GPU is attached (free T4 is enough).
2. Installs Unsloth + dependencies.
3. Logs into Hugging Face (accept the Gemma 3 license once at https://huggingface.co/google/gemma-3-270m-it).
4. Loads `unsloth/gemma-3-270m-it` in QLoRA 4-bit.
5. Fine-tunes 3 epochs on the full CoEdIT (69k rows) — ~8 min on T4.
6. Saves the adapter + exports a ~60 MB q4_k_m GGUF.
7. Mounts Drive and stashes the artifacts so the runtime can die without losing your work.

Adapter ends up at `/content/drive/MyDrive/quill/gemma3-270m-coedit-lora/`. GGUF at `/content/drive/MyDrive/quill/quill-q4_k_m.gguf`.

---

## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Install deps

Unsloth's Colab installer handles torch / xformers / bitsandbytes pinning correctly — don't pip-install them separately.

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<1" peft accelerate bitsandbytes
!pip install -q datasets sacrebleu pyyaml

## 3. Hugging Face login

Paste your HF token (`hf_…`) when prompted. Required because Gemma 3 is license-gated.

You only need to accept the license once: https://huggingface.co/google/gemma-3-270m-it

In [ ]:
from huggingface_hub import login
login()

## 4. Load model (Gemma 3 270M, QLoRA 4-bit)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-270m-it",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # auto: bf16 on Ampere+, fp16 on T4
    load_in_4bit=True,     # QLoRA
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=1337,
    use_rslora=False,
)

## 5. Load + format CoEdIT

In [ ]:
from datasets import load_dataset

GEMMA_CHAT_TEMPLATE = (
    "<start_of_turn>user\n{src}<end_of_turn>\n"
    "<start_of_turn>model\n{tgt}<end_of_turn>"
)

def format_coedit(row):
    return {"text": GEMMA_CHAT_TEMPLATE.format(src=row["src"], tgt=row["tgt"])}

ds = load_dataset("grammarly/coedit")
train_ds = ds["train"].map(format_coedit, remove_columns=ds["train"].column_names)
eval_ds  = ds["validation"].map(format_coedit, remove_columns=ds["validation"].column_names).select(range(1000))

print(f"train: {len(train_ds)} rows  ·  eval: {len(eval_ds)} rows")
print("sample:", train_ds[0]["text"][:200])

## 6. Train (~8 min on T4)

In [ ]:
from trl import SFTConfig, SFTTrainer

OUT_DIR = "/content/checkpoints/gemma3-270m-coedit-lora"
use_bf16 = is_bfloat16_supported()

sft_cfg = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.0,
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    seed=1337,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    optim="adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f"adapter saved to {OUT_DIR}")

## 7. Quick sanity-check the adapter

In [ ]:
FastLanguageModel.for_inference(model)

for src in [
    "Fix the grammaticality: This is an test of the Harper grammer checker.",
    "Improve the grammaticality: I has a apple.",
    "Paraphrase this sentence: Why are you arresting me?",
]:
    prompt = f"<start_of_turn>user\n{src}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    decoded = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    decoded = decoded.split("<end_of_turn>")[0].strip()
    print(f"src : {src}")
    print(f"pred: {decoded}\n")

## 8. Export quantized GGUF (q4_k_m, ~60 MB)

This is the file the desktop shell will load via candle / llama-cpp-rs.

In [ ]:
GGUF_PATH = "/content/checkpoints/quill-q4_k_m.gguf"
model.save_pretrained_gguf(GGUF_PATH, tokenizer, quantization_method="q4_k_m")
!ls -lah {GGUF_PATH}

## 9. Persist to Drive

Runtime can die at any time on free Colab. Copy the artifacts to Drive so you can pull them down to `~/quill/train/checkpoints/` later.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/quill
!cp -r /content/checkpoints/gemma3-270m-coedit-lora /content/drive/MyDrive/quill/
!cp     /content/checkpoints/quill-q4_k_m.gguf      /content/drive/MyDrive/quill/
!ls -lah /content/drive/MyDrive/quill/

Done. Pull the artifacts back to your Mac with `rclone` or the Drive web UI into `~/quill/train/checkpoints/`, then wire the GGUF into the shell.